# Load dim products

In [0]:
# PRODUCT_CATegory         PRODUC_INFO						
#                     product_id-> internal surrogate key 
# category_key",      product_key-> product_category+ additional info
# category",          product_name
# subcategory",       product_cost
# maintenance"        product_line
#                     product_start_date
#                     product_end_date
# product line = Road,Mountain, Sport, Touring
query = """
SELECT
    -- identifiers
    --surrogate key
    ROW_NUMBER() OVER (ORDER BY COALESCE(i.product_key)) as product_sk,
    -- natural key
    i.product_key AS product_nk,
    -- attributes
    i.product_name name,
    c.category category,
    c.subcategory subcategory,
    i.product_line product_family,
    -- prices
    i.product_cost cost,
    --dates
    i.product_start_date start_date,
    i.product_end_date end_date,
    c.maintenance maintenance,
    CURRENT_DATE as load_date

FROM bike_data.silver.crm_product_information i
LEFT JOIN bike_data.silver.erp_prduct_category C
ON SUBSTRING(i.product_key,1,5) = c.category_key
"""
df = spark.sql(query)

In [0]:
df.limit(10).display()

# Write to Gold table

In [0]:
(df.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("bike_data.gold.dim_products"))